# Ensemble multi-modelo — 9105 + 9106 + 9107

Objetivo: promediar las probabilidades de los 3 experimentos ganadores (60 predicciones totales) para explotar la diversidad estructural entre ellos.

**Hipótesis:** los 3 modelos capturan la señal de maneras distintas (FE ratios + HP hardcoded, FE ratios + BO recalibrada, FE ratios + CV temporal). Promediarlos puede reducir el error de manera más efectiva que 60 semillas del mismo modelo.

**Sin re-entrenamiento.** Solo lee las predicciones ya guardadas en disco de cada experimento.

In [ ]:
# --- Setup ---
library(data.table)

# AJUSTAR paths a tu VM si difieren
paths_experimentos <- list(
  "9105" = "/content/buckets/b1/exp/WF9105/semillas",
  "9106" = "/content/buckets/b1/exp/WF9106/semillas",
  "9107" = "/content/buckets/b1/exp/WF9107/semillas"
)

# Las 20 semillas usadas en los 3 experimentos
semillas <- c(
  804043, 653561, 703903, 439693, 665857,
  246319, 719179, 688511, 678859, 759179,
  748567, 319687, 771091, 684007, 514853,
  377749, 329977, 757927, 724837, 216973
)

# Directorio de trabajo para outputs de este ensemble
setwd(paths_experimentos[["9105"]])  # trabajamos desde el dir del 9105
setwd("..")  # subimos un nivel
dir.create("kaggle_ensemble", showWarnings = FALSE)

# Verifico que los directorios existan
for (exp in names(paths_experimentos)) {
  path <- paths_experimentos[[exp]]
  if (!dir.exists(path)) {
    stop("No existe el directorio del experimento ", exp, ": ", path)
  }
  n_files <- length(list.files(path, pattern = "^prediccion_semilla_"))
  cat(exp, ": ", n_files, " archivos en ", path, "\n", sep = "")
}

In [ ]:
# --- Cargar las 60 predicciones (20 por experimento) ---

# Uso el orden de numero_de_cliente del primer archivo como referencia
primer_archivo <- fread(paste0(paths_experimentos[["9105"]],
                               "/prediccion_semilla_", semillas[1], ".txt"))
tb_probs_all <- primer_archivo[, list(numero_de_cliente)]

n_ok <- 0
for (exp in names(paths_experimentos)) {
  path <- paths_experimentos[[exp]]
  for (semilla in semillas) {
    archivo <- paste0(path, "/prediccion_semilla_", semilla, ".txt")
    if (!file.exists(archivo)) {
      warning("Falta archivo: ", archivo)
      next
    }
    tb_ind <- fread(archivo)
    # aseguro el mismo orden que la tabla acumuladora
    tb_ind <- tb_ind[match(tb_probs_all$numero_de_cliente, tb_ind$numero_de_cliente)]
    col_nombre <- paste0("p_", exp, "_", semilla)
    tb_probs_all[, (col_nombre) := tb_ind$prob]
    n_ok <- n_ok + 1
  }
}

cat("Predicciones cargadas:", n_ok, "/60\n")
cat("Columnas de tb_probs_all:", ncol(tb_probs_all), "\n")

In [ ]:
# --- Análisis local: correlación entre los 3 modelos base ---

# Primero promedio dentro de cada experimento (ensemble intra-modelo)
cols_9105 <- grep("^p_9105_", colnames(tb_probs_all), value = TRUE)
cols_9106 <- grep("^p_9106_", colnames(tb_probs_all), value = TRUE)
cols_9107 <- grep("^p_9107_", colnames(tb_probs_all), value = TRUE)

tb_probs_all[, ens_9105 := rowMeans(.SD), .SDcols = cols_9105]
tb_probs_all[, ens_9106 := rowMeans(.SD), .SDcols = cols_9106]
tb_probs_all[, ens_9107 := rowMeans(.SD), .SDcols = cols_9107]

# Correlación entre los 3 ensembles
cat("=== Correlación entre los 3 modelos (ensemble intra por modelo) ===\n")
cor_matrix <- cor(tb_probs_all[, .(ens_9105, ens_9106, ens_9107)])
print(round(cor_matrix, 4))

cat("\nInterpretación:\n")
cat("  Si las 3 correlaciones son muy altas (>0.99): los modelos ven la misma señal\n")
cat("  → el ensemble multi-modelo va a aportar poco.\n")
cat("  Si están en 0.95-0.98: hay diversidad genuina → el ensemble puede ayudar.\n")
cat("  Si están < 0.95: mucha diversidad, riesgo de que el promedio diluya señal.\n")

In [ ]:
# --- Armar 2 versiones de ensemble multi-modelo ---

# ENSEMBLE A: 9105 + 9106 (40 predicciones)
# Los 2 modelos con mejor performance individual en Public
cols_9105_9106 <- c(cols_9105, cols_9106)
tb_probs_all[, ens_9105_9106 := rowMeans(.SD), .SDcols = cols_9105_9106]

# ENSEMBLE B: 9105 + 9106 + 9107 (60 predicciones)
# Los 3 modelos, máxima diversidad
cols_todos <- c(cols_9105, cols_9106, cols_9107)
tb_probs_all[, ens_todos := rowMeans(.SD), .SDcols = cols_todos]

cat("Ensembles armados:\n")
cat("  A: 9105 + 9106 (40 predicciones)\n")
cat("  B: 9105 + 9106 + 9107 (60 predicciones)\n")

# comparación de coincidencia top-2000 entre ensembles
corte_ref <- 2000

get_top <- function(prob_col, k) {
  tb_tmp <- tb_probs_all[, list(numero_de_cliente, p = get(prob_col))]
  setorder(tb_tmp, -p)
  tb_tmp[1:k, numero_de_cliente]
}

top_9105 <- get_top("ens_9105", corte_ref)
top_9106 <- get_top("ens_9106", corte_ref)
top_9107 <- get_top("ens_9107", corte_ref)
top_A <- get_top("ens_9105_9106", corte_ref)
top_B <- get_top("ens_todos", corte_ref)

cat("\n=== Coincidencias en top-", corte_ref, " ===\n", sep = "")
cat("9105 vs 9106:      ", length(intersect(top_9105, top_9106)),
    "(", round(length(intersect(top_9105, top_9106))/corte_ref, 3), ")\n")
cat("9105 vs 9107:      ", length(intersect(top_9105, top_9107)),
    "(", round(length(intersect(top_9105, top_9107))/corte_ref, 3), ")\n")
cat("9106 vs 9107:      ", length(intersect(top_9106, top_9107)),
    "(", round(length(intersect(top_9106, top_9107))/corte_ref, 3), ")\n")
cat("En los 3 modelos:  ",
    length(Reduce(intersect, list(top_9105, top_9106, top_9107))),
    "(", round(length(Reduce(intersect, list(top_9105, top_9106, top_9107)))/corte_ref, 3), ")\n")
cat("Ensemble A vs B:   ", length(intersect(top_A, top_B)),
    "(", round(length(intersect(top_A, top_B))/corte_ref, 3), ")\n")

In [ ]:
# --- Submit del Ensemble A (9105 + 9106) a Kaggle ---

competencia <- "data-mining-junior-2026-a"
cortes <- seq(1800, 2400, by = 100)

# Preparo tb_A ordenada
tb_A <- tb_probs_all[, list(numero_de_cliente, prob = ens_9105_9106)]
setorder(tb_A, -prob)

for (envios in cortes) {
  tb_A[, Predicted := 0L]
  tb_A[1:envios, Predicted := 1L]

  archivo <- paste0("./kaggle_ensemble/KAensembleA_9105_9106_", envios, ".csv")
  fwrite(tb_A[, list(numero_de_cliente, Predicted)], file = archivo, sep = ",")

  linea <- paste(
    "kaggle competitions submit",
    "-c", competencia,
    "-f", archivo,
    paste0("-m 'Ensemble A: 9105+9106 (40) envios=", envios, "'")
  )

  cat(format(Sys.time(), "%X"), " - submit Ensemble A envios=", envios, "\n", sep = "")
  salida <- system(linea, intern = TRUE)
  Sys.sleep(30)
  cat(salida, "\n")
}

In [ ]:
# --- Submit del Ensemble B (9105 + 9106 + 9107) a Kaggle ---

tb_B <- tb_probs_all[, list(numero_de_cliente, prob = ens_todos)]
setorder(tb_B, -prob)

for (envios in cortes) {
  tb_B[, Predicted := 0L]
  tb_B[1:envios, Predicted := 1L]

  archivo <- paste0("./kaggle_ensemble/KAensembleB_9105_9106_9107_", envios, ".csv")
  fwrite(tb_B[, list(numero_de_cliente, Predicted)], file = archivo, sep = ",")

  linea <- paste(
    "kaggle competitions submit",
    "-c", competencia,
    "-f", archivo,
    paste0("-m 'Ensemble B: 9105+9106+9107 (60) envios=", envios, "'")
  )

  cat(format(Sys.time(), "%X"), " - submit Ensemble B envios=", envios, "\n", sep = "")
  salida <- system(linea, intern = TRUE)
  Sys.sleep(30)
  cat(salida, "\n")
}

cat("\nTodos los submits completados. Compará los Public de A y B contra 9105, 9106 y 9107.\n")